# DATA 202 Homework 6: Data Wrangling


In [2]:
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar
import datetime

# Source Data

## Capital Bikeshare Rides Data

Download the [2011 trip data](https://s3.amazonaws.com/capitalbikeshare-data/2011-capitalbikeshare-tripdata.zip) from [Capital Bikeshare](https://www.capitalbikeshare.com/system-data). Don't need to unzip the ZIP file; Pandas will handle it:

In [3]:
rides = pd.read_csv("2011-capitalbikeshare-tripdata.zip")
rides.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1226767 entries, 0 to 1226766
Data columns (total 9 columns):
Duration                1226767 non-null int64
Start date              1226767 non-null object
End date                1226767 non-null object
Start station number    1226767 non-null int64
Start station           1226767 non-null object
End station number      1226767 non-null int64
End station             1226767 non-null object
Bike number             1226767 non-null object
Member type             1226767 non-null object
dtypes: int64(3), object(6)
memory usage: 84.2+ MB


In [6]:
print('{:,d}'.format(len(rides)))
rides.head()

1,226,767


,Duration,Start date,End date,Start station number,Start station,End station number,End station,Bike number,Member type
0,3548,2011-01-01 00:01:29,2011-01-01 01:00:37,31620,5th & F St NW,31620,5th & F St NW,W00247,Member
1,346,2011-01-01 00:02:46,2011-01-01 00:08:32,31105,14th & Harvard St NW,31101,14th & V St NW,W00675,Casual
2,562,2011-01-01 00:06:13,2011-01-01 00:15:36,31400,Georgia & New Hampshire Ave NW,31104,Adams Mill & Columbia Rd NW,W00357,Member
3,434,2011-01-01 00:09:21,2011-01-01 00:16:36,31111,10th & U St NW,31503,Florida Ave & R St NW,W00970,Member
4,233,2011-01-01 00:28:26,2011-01-01 00:32:19,31104,Adams Mill & Columbia Rd NW,31106,Calvert & Biltmore St NW,W00346,Casual


Let's remove some columns we don't need, to save memory.

In [7]:
del rides["Start station"], rides["End station"]

## Holidays

The following code gets us a table of federal holidays. Please run it without changing it.

In [8]:
# Run this code unchanged.
holidays = pd.DataFrame({
    'date': USFederalHolidayCalendar().holidays(datetime.date(2011,1,1), datetime.date(2015,12,31)).date,
    'is_holiday': True})
holidays.head()

,date,is_holiday
0,2011-01-17,True
1,2011-02-21,True
2,2011-05-30,True
3,2011-07-04,True
4,2011-09-05,True


## Weather Data
Our main goal will be to get the hourly temperature data.

The original wranglers used a weather data source that does not seem to provide downloadable data anymore. But we can use the US government's records. They're in a cumbersome format, which will provide us an excuse to practice some **data cleaning**!

First challenge is where to find the data. Here's how we solved this hard problem:

NOAA's [Integrated Surface Database](https://www.ncdc.noaa.gov/data-access/land-based-station-data/land-based-datasets) provides weather data from all over the country. But how to use it? There's a "Find a Station" tool, but it's confusing how to use the results. https://www.ncdc.noaa.gov/data-access/land-based-station-data/station-metadata has a link to a [station list file](ftp://ftp.ncdc.noaa.gov/pub/data/noaa/isd-history.txt). Searching that, it looks like the code for Reagan Airport is 724050 13743. So the file is
https://www.ncei.noaa.gov/data/global-hourly/access/2011/72405013743.csv

Poking around in that site revealed two documents that look very important:
- https://www.ncei.noaa.gov/data/global-hourly/doc/isd-format-document.pdf
- https://www.ncei.noaa.gov/data/global-hourly/doc/CSV_HELP.pdf



In [9]:
# Run this to load the file directly from the NOAA website.
# You may want to make a local copy and read it in from there instead.
weather = pd.read_csv("https://www.ncei.noaa.gov/data/global-hourly/access/2011/72405013743.csv")

/opt/anaconda/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3057: DtypeWarning: Columns (43,47,51,55) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [10]:
print(len(weather))
weather.head()

14558


,STATION,DATE,SOURCE,LATITUDE,LONGITUDE,ELEVATION,NAME,REPORT_TYPE,CALL_SIGN,QUALITY_CONTROL,...,OC1,OD1,OE1,OE2,OE3,RH1,RH2,RH3,REM,EQD
0,72405013743,2011-01-01T00:00:00,4,38.8472,-77.03454,3.0,"WASHINGTON REAGAN NATIONAL AIRPORT, VA US",FM-12,KDCA,V020,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SYN092AAXX 01004 72405 32966 21704 10056 2101...,NaN
1,72405013743,2011-01-01T00:52:00,7,38.8472,-77.03454,3.0,"WASHINGTON REAGAN NATIONAL AIRPORT, VA US",FM-15,KDCA,V030,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MET10012/31/10 19:52:03 METAR KDCA 010052Z 000...,NaN
2,72405013743,2011-01-01T01:52:00,7,38.8472,-77.03454,3.0,"WASHINGTON REAGAN NATIONAL AIRPORT, VA US",FM-15,KDCA,V030,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MET10012/31/10 20:52:03 METAR KDCA 010152Z 000...,NaN
3,72405013743,2011-01-01T02:52:00,7,38.8472,-77.03454,3.0,"WASHINGTON REAGAN NATIONAL AIRPORT, VA US",FM-15,KDCA,V030,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MET10612/31/10 21:52:03 METAR KDCA 010252Z 180...,NaN
4,72405013743,2011-01-01T03:00:00,4,38.8472,-77.03454,3.0,"WASHINGTON REAGAN NATIONAL AIRPORT, VA US",FM-12,KDCA,V020,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SYN076AAXX 01034 72405 32966 41803 10039 2100...,NaN


# Data Wrangling

## 1. Extract `date` and `hour`

In [ ]:
rides['start'] = pd.to_datetime(rides['Start date'])
rides['start'].iloc[0]

In [ ]:
rides['date'] = rides['start'].dt.date#strftime("%Y-%m-%d")
rides['hour'] = rides['start'].dt.hour

## 2. Filter to include only rides by Members
You'll end up with a Series with a hierarchical index; remember that the "get out of jail card" is `.to_frame(name="NAME_GOES_HERE").reset_index()`.

In [ ]:
# your code here
...

## ... more data wrangling...

## Yay, we're done!

In [ ]:
assert len(merged_data) > 365 * 23
assert 'date' in merged_data.columns
assert 'hour' in merged_data.columns
assert 'is_holiday' in merged_data.columns
assert 'temp_C' in merged_data.columns
assert 'rides' in merged_data.columns
assert len(merged_data.dropna()) == len(merged_data)